# Очистка данных: Deals

In [28]:
import os
import re
import datetime
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h


# Пути к данным
RAW_DATA_DIR = os.path.join('..', 'Sources')
CLEANED_DIR  = os.path.join('..', 'data', 'cleaned')

DEALS_INPUT = os.path.join(RAW_DATA_DIR, 'Deals (Done).xlsx')
DEALS_OUTPUT = os.path.join(CLEANED_DIR, 'deals_clean.pkl')
MAPPING_INPUT = os.path.join(CLEANED_DIR, 'contacts_mapping.pkl')

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

In [29]:
def time_to_seconds(t):
    """Конвертирует datetime.time или число → int (секунды) или pd.NA."""
    if isinstance(t, datetime.time):
        return int(t.hour * 3600 + t.minute * 60 + t.second)
    
    try:
        val = float(t)
        if np.isfinite(val):
            return int(round(val))
    except (ValueError, TypeError):
        pass
    
    return pd.NA

## Загрузка и первичный осмотр

In [30]:
# Contact Name и Id читаем как str, чтобы избежать
# потери точности при промежуточном float64 (проявляется при наличии NaN в колонке)
df = pd.read_excel(DEALS_INPUT, dtype={'Contact Name': str, 'Id': str})

# Переименование столбцов в snake_case
df.columns = [h.to_snake(c) for c in df.columns]

# Переименование contact_name в contact_id для единообразия с другими таблицами
df = df.rename(columns={'contact_name': 'contact_id'})

# Используем df_de_raw, чтобы не накапливать ошибки при повторных запусках 
# для нормализации уровня немецкого языка
df_de_raw = df['level_of_deutsch']

n_before = len(df)

print(f'Форма: {df.shape}')

h.descr_df(df, include='all', show_stats=False, show_sample_rows=False)

Форма: (21595, 23)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений
0,id,str,21593,2,21593
1,deal_owner_name,str,21564,31,27
2,closing_date,str,14645,6950,359
3,quality,str,19340,2255,6
4,stage,str,21593,2,13
5,lost_reason,str,16124,5471,21
6,page,str,21593,2,34
7,campaign,str,16067,5528,154
8,sla,object,15533,6062,13357
9,content,str,14147,7448,187


In [31]:
# Проверка уникальности технического ID
ids_counts = df['id'].nunique()
print(f'Уникальных ID сделок: {ids_counts} ({"все ID уникальны" if ids_counts == n_before else "есть дубликаты по ID!"})')

# Поиск бизнес-дубликатов по ключевым полям: клиент, время создания, продукт, сумма оплаты
# Убираем offer_total_amount из ключей, так как он может варьироваться или быть пустым
BUSINESS_KEYS = ['contact_id', 'created_time', 'product', 'initial_amount_paid']

# Проверяем наличие колонок перед поиском дублей
search_cols = [c for c in BUSINESS_KEYS if c in df.columns]
biz_dupes = df.duplicated(subset=search_cols).sum()
lost_dupes = df[df['lost_reason'].astype(str).str.contains('дубликат|duplicate', case=False, na=False)]

print(f'Обнаружено бизнес-дубликатов по ключам {search_cols}: {biz_dupes}')
print(f"Найдено сделок с lost_reason='дубликат': {len(lost_dupes)}")

Уникальных ID сделок: 21593 (есть дубликаты по ID!)
Обнаружено бизнес-дубликатов по ключам ['contact_id', 'created_time', 'product', 'initial_amount_paid']: 24
Найдено сделок с lost_reason='дубликат': 1771


In [5]:
# Удаляем технический шум (полностью пустые строки без Id)
df = df.dropna(subset=['id']).reset_index(drop=True)
print(f'Строк после удаления пустых Id: {len(df)}')

Строк после удаления пустых Id: 21593


## Дедупликация

In [32]:
# Удаление записей с явной пометкой 'дубликат' в CRM (Пометил менеджер)
before_crm = len(df)
df = df[~df['lost_reason'].astype(str).str.contains('дубликат|duplicate', case=False, na=False)]
print(f'Удалено записей с lost_reason="дубликат": {before_crm - len(df)}')

# Удаление полных дубликатов (если вдруг остались)
before_full = len(df)
df = df.drop_duplicates()
print(f'Удалено полных дубликатов: {before_full - len(df)}')

# Удаление бизнес-дубликатов (ключи совпадают, ID разные)
# Состав ключей: клиент, время создания, продукт, сумма оплаты
before_biz = len(df)
df = df.drop_duplicates(subset=['contact_id', 'created_time', 'product', 'initial_amount_paid'], keep='last')
print(f'Удалено найденных нами бизнес-дубликатов: {before_biz - len(df)}')

print(f'\nИтого строк после дедупликации: {len(df)}')

Удалено записей с lost_reason="дубликат": 1771
Удалено полных дубликатов: 0
Удалено найденных нами бизнес-дубликатов: 8

Итого строк после дедупликации: 19816


In [33]:
# Очистка названий городов (City)
# Убираем лишние детали и исправляем написание для корректного геокодирования
city_fixes = {
    'Bad Wildbad im Schwarzwald': 'Bad Wildbad',
    'Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a': 'Gdansk',
    'Alzenau in Unterfranken': 'Alzenau',
    '-': 'Unknown'
}

# Список ключей для вывода (все кроме '-')
other_fixes = [k for k in city_fixes.keys() if k != '-']

# Считаем количество вхождений ДО замены
n_fixes_other = df['city'].isin(other_fixes).sum()
n_dash = (df['city'] == '-').sum()

# Собираем статистику по каждому исправлению (сколько раз встретилось в данных)
stats_list = []
for old in other_fixes:
    count = (df['city'] == old).sum()
    if count > 0:
        stats_list.append(f"  '{old}' -> '{city_fixes[old]}' ({count} шт.)")

df['city'] = df['city'].replace(city_fixes)

print(f'Найдено и исправлено сложных адресов: {n_fixes_other}')
print(f'Заменено тире ("-") на "Unknown": {n_dash}')

if stats_list:
    print("\nДетализация исправлений:")
    for line in stats_list:
        print(line)
else:
    print("\nСложных адресов из списка city_fixes в текущих данных не обнаружено.")

Найдено и исправлено сложных адресов: 5
Заменено тире ("-") на "Unknown": 348

Детализация исправлений:
  'Bad Wildbad im Schwarzwald' -> 'Bad Wildbad' (2 шт.)
  'Poland , Gdansk , Al. Grunwaldzka 7, ap. 1a' -> 'Gdansk' (1 шт.)
  'Alzenau in Unterfranken' -> 'Alzenau' (2 шт.)


## 6. Id и Contact Id: object → Int64 (безопасно)

In [35]:
# id и contact_id читаем как str. конвертируем в Int64 напрямую через Python int(),
# минуя float64-промежуток, который округляет 19-значные числа.
# Заодно убираем '.0' / '.00', которые Excel может добавить при сохранении числа как float.

df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')
df['contact_id'] = pd.array([h.str_to_int64(v) for v in df['contact_id']], dtype='Int64')

print(f'Пропусков в id (тип {df["id"].dtype}):        {df["id"].isna().sum()}')
print(f'Пропусков в contact_id (тип {df["contact_id"].dtype}): {df["contact_id"].isna().sum()}')

Пропусков в id (тип Int64):        1
Пропусков в contact_id (тип Int64): 48


## Типы данных: даты

In [36]:
# Created Time: '21.06.2024 15:30'  → datetime
df['created_time'] = pd.to_datetime(df['created_time'], dayfirst=True, errors='coerce')

# Closing Date: '21.06.2024' → datetime (NaT = сделка ещё открыта)
df['closing_date'] = pd.to_datetime(df['closing_date'], dayfirst=True, errors='coerce')

# 1. Сырая длительность (с сохранением ОТРИЦАТЕЛЬНЫХ значений для анализа менеджеров)
# Здесь мы только исправляем техническую ошибку -1 (совпадение дат), так как это не бизнес-аномалия
df['duration_raw'] = (df['closing_date'] - df['created_time']).dt.days
same_day_mask = (df['duration_raw'] == -1) & (df['closing_date'].dt.date == df['created_time'].dt.date)
df.loc[same_day_mask, 'duration_raw'] = 0

# 2. Обработка дат для итоговой аналитики (deal_duration_days)
# Создаем копию даты закрытия для манипуляций
df['closing_date_fixed'] = df['closing_date'].copy()

# ПРАВИЛО 1: Убираем реальные аномалии (дата закрытия РАНЬШЕ даты создания)
# Приравниваем к созданию, чтобы не искажать среднее отрицательными числами, 
# но в duration_raw они останутся для менеджеров.
anom_mask = (df['closing_date_fixed'] < df['created_time'])
df.loc[anom_mask, 'closing_date_fixed'] = df.loc[anom_mask, 'created_time']

# Считаем итоговую длительность
# ВАЖНО: Если closing_date был NaT (например, для Lost), deal_duration_days тоже станет NaT.
# Это корректно: мы не знаем, когда сделка была фактически потеряна, если CRM не сохранила дату.
df['deal_duration_days'] = (df['closing_date_fixed'] - df['created_time']).dt.days

# Исправляем техническую ошибку -1 (совпадение дат) для итоговой колонки
same_day_final = (df['deal_duration_days'] == -1) & (df['closing_date_fixed'].dt.date == df['created_time'].dt.date)
df.loc[same_day_final, 'deal_duration_days'] = 0

# Удаляем временную колонку
df = df.drop(columns=['closing_date_fixed'])

print('created_time:', df['created_time'].dtype, '| NaT:', df['created_time'].isna().sum())
print('closing_date:', df['closing_date'].dtype, '| NaT:', df['closing_date'].isna().sum())
print('duration_raw (сырая + фикс -1):', df['duration_raw'].dtype)
print('deal_duration_days (исправленная):', df['deal_duration_days'].dtype)

created_time: datetime64[us] | NaT: 1
closing_date: datetime64[us] | NaT: 6681
duration_raw (сырая + фикс -1): float64
deal_duration_days (исправленная): float64


## SLA: время ответа → секунды

> `SLA` хранит объекты `datetime.time` (hh:mm:ss). Для анализа удобнее хранить как **целое число секунд**.

In [37]:
print('SLA — Обработка данных')

# 1. Сначала превращаем всё в float64 через pd.to_numeric
# Это гарантирует, что <NA> превратится в np.nan, который понимает .round()
sla_raw = pd.to_numeric(df['sla'].apply(time_to_seconds), errors='coerce') / 60

# 2. Создаем sla и sla_filled
df['sla'] = sla_raw.round().astype('Int32')
df['sla_filled'] = sla_raw.copy()

# Удаляем аномалии (> 24 часов)
MAX_SLA_MINUTES = 24 * 60
mask_out = (df['sla_filled'] > MAX_SLA_MINUTES)
if mask_out.any():
    print(f"В 'sla_filled' сброшено аномалий (>24ч): {mask_out.sum()}")
    df.loc[mask_out, 'sla_filled'] = np.nan

# Восполняем пропуски
if 'deal_owner_name' in df.columns:
    # Важно: медиана возвращает float, fillna на float64 работает корректно
    medians = df.groupby('deal_owner_name', observed=True)['sla_filled'].transform('median')
    glob_med = df['sla_filled'].median()
    
    n_nan_before = df['sla_filled'].isna().sum()
    df['sla_filled'] = df['sla_filled'].fillna(medians).fillna(glob_med)
    n_filled = n_nan_before - df['sla_filled'].isna().sum()
    print(f"В 'sla_filled' восполнено пропусков: {n_filled}")

# Окончательно приводим к Int32
df['sla_filled'] = df['sla_filled'].round().astype('Int32')

print(f'\n--- Итог по колонкам SLA (в целых минутах) ---')
print(f"1. 'sla' (сырое): {df['sla'].dtype} | NaN = {df['sla'].isna().sum()}")
print(f"2. 'sla_filled' (дозап.): {df['sla_filled'].dtype} | NaN = {df['sla_filled'].isna().sum()}")

if df["sla_filled"].notna().any():
    print(f"\nСреднее 'sla_filled' (экономика): {df['sla_filled'].mean():.1f} мин")

SLA — Обработка данных
В 'sla_filled' восполнено пропусков: 6579

--- Итог по колонкам SLA (в целых минутах) ---
1. 'sla' (сырое): Int32 | NaN = 6579
2. 'sla_filled' (дозап.): Int32 | NaN = 0

Среднее 'sla_filled' (экономика): 362.8 мин


## 5. Числовые поля: очистка сумм

In [38]:
for col in ['initial_amount_paid', 'offer_total_amount']:
    df[col] = h.clean_amount(df[col])

# Преобразование в Int32 с поддержкой NaN
for col in ['course_duration', 'months_of_study']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int32')


## 7. Level of Deutsch: нормализация

> Поле содержит 215 уникальных значений: смесь кириллицы и латиницы (например `а2` vs `A2`, `б1` vs `B1`),
> а также свободный текст (адреса, фразы). Приводим к стандарту CEFR (A0–C2), остальное → `Unknown`.

In [39]:
# Список всех значений, которые не соответствуют паттерну [A1-C2] после очистки кириллицы
CYR_TO_LAT = str.maketrans('абвсАБВС', 'abvcABVC')

# ТАБЛИЦА ЗАМЕН: 
LEVEL_MAP = {
    'а': 'a', 'А': 'A',
    'б': 'b', 'Б': 'B',
    'в': 'b', 'В': 'B',
    'с': 'c', 'С': 'C'
}

def normalize_deutsch(value):
    if pd.isna(value):
        return pd.NA
    
    orig_s = str(value).strip()
    s_lower = orig_s.lower()
    
    # 1. Специальные случаи для A0
    a0_exact = ['0', 'no', 'none', '?', '-', 'нет', 'a'] # A без цифры -> A0
    a0_keywords = ['никакой', 'нулевой', 'не учил', 'не учила', 'anfanger', 'beginner', 'начальный']
    if s_lower in a0_exact or any(keyword in s_lower for keyword in a0_keywords):
        return 'A0'
        
    # 2. Специальные случаи для других уровней
    if s_lower == 'в': return 'B1'
    if s_lower == 'f2': return 'A2'
    if s_lower == 'c': return 'C1'
    
    # 3. ПОИСК УРОВНЕЙ: Любая буква [AaBbCcАаБбВвСс] + цифра [012]
    match = re.search(r'([AaBbCcАаБбВвСс][0-2])', orig_s)
    if match:
        found = match.group(1)
        letter = found[0]
        digit = found[1]
        letter_lat = LEVEL_MAP.get(letter, letter).upper()
        return f"{letter_lat}{digit}"
    
    # 4. Маппинг остальных ключевых слов
    words_map = {
        'intermediate': 'B1', 'средний': 'B1',
        'advanced': 'C1'
    }
    for word, level in words_map.items():
        if word in s_lower:
            return level
            
    return 'Unclear'

# Применяем очистку
# Используем df_raw, чтобы не накапливать ошибки при повторных запусках
# df_raw = pd.read_excel(DEALS_INPUT)
df['level_of_deutsch'] = df_de_raw.apply(normalize_deutsch)

# Анализ оставшихся нестандартных значений (для проверки)
def get_non_standard(value):
    if pd.isna(value) or value == 'Unclear': return value
    # Если значение уже в стандарте A1-C2, возвращаем None
    if re.search(r'\b([AaBbCc][012])\b', str(value)):
        return None
    return value

non_std_count = (df['level_of_deutsch'] == 'Unclear').sum()
print(f"Всего строк с нераспознанным уровнем (Unclear): {non_std_count}")

print("\n--- ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ УРОВНЕЙ ---")
display(df['level_of_deutsch'].value_counts().to_frame())

Всего строк с нераспознанным уровнем (Unclear): 15

--- ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ УРОВНЕЙ ---


,count
level_of_deutsch,
B1,816
B2,169
A2,150
A0,40
C1,28
A1,25
Unclear,15
C2,3


## 6. Группировка стадий (Funnel)

Для построения воронки продаж и анализа конверсии мы объединяем 13 детальных стадий CRM в 4 укрупненные бизнес-группы:
1. **Marketing/Lead** — новые регистрации и потенциальные интересы.
2. **Active Sales** — стадия переговоров, консультаций и пробных периодов.
3. **Won/Paid** — успешное завершение сделки (оплата).
4. **Lost** — закрытые сделки без оплаты.

In [40]:
# ГРУППИРОВКА СТАДИЙ
# Цель: упростить воронку до 4 бизнес-этапов

STAGE_GROUPS = {
    'New Lead': 'Marketing/Lead',
    'Registered on Webinar': 'Marketing/Lead',
    'Registered on Offline Day': 'Marketing/Lead',
    'Need To Call': 'Active Sales',
    'Need to Call - Sales': 'Active Sales',
    'Need a consultation': 'Active Sales',
    'Qualificated': 'Active Sales',
    'Test Sent': 'Active Sales',
    'Call Delayed': 'Active Sales',
    'Waiting For Payment': 'Active Sales',
    'Free Education': 'Active Sales',
    'Payment Done': 'Won/Paid',
    'Lost': 'Lost'
}

# Создание новой колонки
df['stage_group'] = df['stage'].map(STAGE_GROUPS).fillna('Other')

# 2. Проверка результатов
print("Распределение стадий по группам:")
display(df['stage_group'].value_counts().to_frame())

Распределение стадий по группам:


,count
stage_group,
Lost,13996
Active Sales,2808
Marketing/Lead,2155
Won/Paid,856
Other,1


## 7. Восполнение пропусков (Backfill) на основе Contact Name

> Если для одного и того же клиента (`Contact Name`) в разных сделках заполнены разные поля (Source, City и т.д.), 
> мы можем «протянуть» эти значения на пустые строки этого же клиента.

In [21]:
# Поля для заполнения
COLS_TO_FILL = ['source', 'campaign', 'city', 'level_of_deutsch', 'deal_owner_name']
COLS_CHECK = COLS_TO_FILL + ['course_duration', 'offer_total_amount']

df['is_buyer'] = (df['initial_amount_paid'] > 0) & (df['months_of_study'] > 0)

# Приводим колонки к object перед восполнением
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df[col].astype(object)

# Считаем "пропуски" до восполнения. 
# Для финансовых полей считаем 0 как пропуск, так как мы их будем восполнять.
missing_before = df[COLS_TO_FILL].isnull().sum()
missing_before['course_duration'] = df['course_duration'].isna().sum()
missing_before['offer_total_amount'] = (df['offer_total_amount'].isna() | (df['offer_total_amount'] == 0)).sum()

# Backfill по contact_id (протягиваем данные между разными сделками одного клиента)
df = df.sort_values(['contact_id', 'created_time'])
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df.groupby('contact_id', group_keys=False)[col].apply(lambda x: x.ffill().bfill())

# Восполнение deal_owner_name из контактов
CONTACTS_CLEAN = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')
if os.path.exists(CONTACTS_CLEAN):
    contacts = pd.read_pickle(CONTACTS_CLEAN)
    contacts['id'] = contacts['id'].astype('Int64')
    contact_owner_map = contacts.drop_duplicates('id').set_index('id')['contact_owner_name']
    
    mask_isna = df['deal_owner_name'].isna()
    df.loc[mask_isna, 'deal_owner_name'] = df.loc[mask_isna, 'contact_id'].map(contact_owner_map)

# Восполнение course_duration и offer_total_amount по продукту (Mode / Median)
if 'product' in df.columns:
    # Ограничиваем выборку только записями, где продукт не 'Unknown'
    valid_prod_mask = (df['product'].notna()) & (df['product'] != 'Unknown')
    
    # Длительность курса (Мода)
    prod_duration_map = (
        df[valid_prod_mask]
        .groupby('product', observed=True)['course_duration']
        .apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )
    
    # Полная стоимость оффера (Медиана)
    # Берем медиану только из тех строк, где сумма > 0
    prod_offer_map = (
        df[valid_prod_mask & (df['offer_total_amount'] > 0)]
        .groupby('product', observed=True)['offer_total_amount']
        .median()
    )

    # Применяем восполнение только для NaN
    mask_dur = df['course_duration'].isna()
    df.loc[mask_dur, 'course_duration'] = df.loc[mask_dur, 'product'].map(prod_duration_map)
    
    mask_offer = (df['offer_total_amount'].isna()) | (df['offer_total_amount'] == 0)
    df.loc[mask_offer, 'offer_total_amount'] = df.loc[mask_offer, 'product'].map(prod_offer_map)

# Считаем итоги
missing_after = df[COLS_TO_FILL].isnull().sum()
missing_after['course_duration'] = df['course_duration'].isna().sum()
missing_after['offer_total_amount'] = (df['offer_total_amount'].isna() | (df['offer_total_amount'] == 0)).sum()

filled = missing_before - missing_after

print('\nДинамика восполнения пропусков (0 в офферах теперь считаются как пропуски):')
display(pd.DataFrame({
    'Было (NaN или 0)': missing_before,
    'Восполнено': filled,
    'Осталось пропусков': missing_after
}))

# УДАЛЕНО: Больше не подставляем дату создания для потерянных сделок без даты.
# Мы понимаем, что Lost-сделка могла длиться долго, и приравнивание к 0 искажает аналитику.
# lost_no_date = (df['stage'] == 'Lost') & (df['closing_date'].isna())
# df.loc[lost_no_date, 'closing_date'] = df.loc[lost_no_date, 'created_time']
# print(f"Заполнена дата закрытия (равна дате создания) для {lost_no_date.sum()} потерянных сделок.")


Динамика восполнения пропусков (0 в офферах теперь считаются как пропуски):


,Было (NaN или 0),Восполнено,Осталось пропусков
source,0,-47,47
campaign,4235,810,3425
city,17307,607,16700
level_of_deutsch,18570,329,18241
deal_owner_name,29,-40,69
course_duration,16282,0,16282
offer_total_amount,16546,277,16269


In [17]:
# Поля, где пропуск = отсутствие информации → заполняем 'Unknown'
FILL_UNKNOWN = ['quality', 'lost_reason', 'source', 'campaign', 'content', 'term',
                'payment_type', 'product', 'education_type', 'city', 'level_of_deutsch', 'deal_owner_name']

for col in FILL_UNKNOWN:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            if isinstance(df[col].dtype, pd.CategoricalDtype):
                if 'Unknown' not in df[col].cat.categories:
                    df[col] = df[col].cat.add_categories('Unknown')
            else:
                df[col] = df[col].astype(object)
            
            df[col] = df[col].fillna('Unknown')
            print(f'{col}: заполнено {n_miss} пропусков → "Unknown"')

# Числовые поля (длительность, сумма оффера, месяцы обучения) → заполняем 0

FILL_ZERO = ['course_duration', 'initial_amount_paid', 'months_of_study', 'offer_total_amount']

for col in FILL_ZERO:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            df[col] = df[col].fillna(0.0)
            print(f'{col}: заполнено {n_miss} пропусков → 0.0')

print("\nФинальный статус пропусков в ключевых колонках:")
remaining = df.isnull().sum()
display(
    pd.DataFrame({'Пропуски': remaining, '%': (remaining / len(df) * 100).round(2)})
    .query('Пропуски > 0')
    .sort_values('Пропуски', ascending=False)
)

quality: заполнено 2230 пропусков → "Unknown"
lost_reason: заполнено 5463 пропусков → "Unknown"
source: заполнено 47 пропусков → "Unknown"
campaign: заполнено 3424 пропусков → "Unknown"
content: заполнено 6021 пропусков → "Unknown"
term: заполнено 7716 пропусков → "Unknown"
payment_type: заполнено 19332 пропусков → "Unknown"
product: заполнено 16278 пропусков → "Unknown"
education_type: заполнено 16559 пропусков → "Unknown"
city: заполнено 16698 пропусков → "Unknown"
level_of_deutsch: заполнено 18240 пропусков → "Unknown"
deal_owner_name: заполнено 47 пропусков → "Unknown"
course_duration: заполнено 16282 пропусков → 0.0
initial_amount_paid: заполнено 15771 пропусков → 0.0
months_of_study: заполнено 18977 пропусков → 0.0
offer_total_amount: заполнено 16269 пропусков → 0.0

Финальный статус пропусков в ключевых колонках:


,Пропуски,%
closing_date,6680,33.71
duration_raw,6680,33.71
deal_duration_days,6680,33.71
sla,6578,33.20
is_buyer,2398,12.10
contact_id,47,0.24


In [18]:
# ПРЕОБРАЗОВАНИЕ ТИПОВ
# Оптимизируем типы данных для уменьшения потребления памяти и удобства анализа

# 1. Категориальные поля
CAT_COLS = [
    'stage', 'stage_group', 'deal_owner_name', 'product', 
    'quality', 'source', 'campaign', 'city', 'level_of_deutsch',
    'payment_type', 'page', 'lost_reason', 'term', 'content'
]

for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 2. Поля с ограниченным набором значений
if 'education_type' in df.columns:
    df['education_type'] = df['education_type'].astype('category')

# 3. Числовые типы
# course_duration и months_of_study могут содержать NaN, поэтому используем Int32
INT_COLS = ['course_duration', 'months_of_study']
for col in INT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int32')

# 4. Флаг покупателя: преобразуем в bool (NaN станет False)
df['is_buyer'] = ((df['initial_amount_paid'].fillna(0) > 0) & (df['months_of_study'].fillna(0) > 0)).astype(bool)


In [ ]:
# Применяем маппинг дублей контактов (сформирован в 01_cleaning_contacts)
if os.path.exists(MAPPING_INPUT):
    contact_mapping = pd.read_pickle(MAPPING_INPUT)
    affected = df['contact_id'].isin(contact_mapping.keys()).sum()
    df['contact_id'] = df['contact_id'].replace(contact_mapping.to_dict())
    print(f"Маппинг контактов применён: {affected} сделок перепривязаны к мастер-контактам.")
else:
    print("Файл contacts_mapping.pkl не найден. Сначала выполните 01_cleaning_contacts.")

os.makedirs(os.path.dirname(DEALS_OUTPUT), exist_ok=True)
df.to_pickle(DEALS_OUTPUT)
df.to_excel(DEALS_OUTPUT.replace('.pkl', '.xlsx'))


Маппинг контактов применён: 26 сделок перепривязаны к мастер-контактам.
Сохранено: ../data/cleaned/deals_clean.pkl


,Метрика,Значение
0,Строк исходно,21595
1,Строк после очистки,19815
2,Удалено дубликатов,1780
3,Уникальных сделок (id),19815
4,Диапазон created_time,2023-07-03 → 2024-06-21
5,stage (уникальных),13
6,"Медиана SLA, мин",239.00
7,Пропуски после заполнения,26665


### Итоговый осмотр

In [ ]:
h.descr_df(df, include='all', show_stats=True, show_sample_rows=True, show_quartiles=True)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,Int64,19815,0,19815,5805028000005176025,5805028000005168037,5805028000005174051,5805028000000921600.00,5805028000030290944.00,5805028000030272512.00,5805028000056893440.00
1,deal_owner_name,category,19815,0,28,John Doe,John Doe,John Doe,<NA>,<NA>,<NA>,<NA>
2,closing_date,datetime64[us],13135,6680,357,NaT,NaT,NaT,<NA>,<NA>,<NA>,<NA>
3,quality,category,19815,0,6,Unknown,Unknown,Unknown,<NA>,<NA>,<NA>,<NA>
4,stage,category,19815,0,13,Registered on Webinar,Registered on Webinar,Registered on Webinar,<NA>,<NA>,<NA>,<NA>
5,lost_reason,category,19815,0,21,Unknown,Unknown,Unknown,<NA>,<NA>,<NA>,<NA>
6,page,category,19815,0,33,/workshop,/workshop,/workshop,<NA>,<NA>,<NA>,<NA>
7,campaign,category,19815,0,152,web2408_DE,web2408_DE,web2408_DE,<NA>,<NA>,<NA>,<NA>
8,sla,Int32,13237,6578,1407,<NA>,<NA>,<NA>,0.00,417.94,239.00,1438.00
9,content,category,19815,0,185,Unknown,Unknown,Unknown,<NA>,<NA>,<NA>,<NA>


Объем памяти: 2.25 MB


## Сохранение

In [ ]:
# ПОДГОТОВКА ЕДИНОГО ФАЙЛА ДЛЯ ОБНОВЛЕНИЯ КОНТАКТОВ (01_contacts)
# Собираем всё в один DataFrame: 
# Признак Buyer (был ли факт перехода: Оплата > 0, деньги получены И Обучение > 0)

BUYERS_INFO_PATH = os.path.join('..', 'data', 'cleaned', 'buyers_info.pkl')

# Агрегация по всем сделкам 
# Ищем самую первую активность (любая сделка, включая Lead/Lost)
reg_fixes_all = (
    df.groupby('contact_id')
    .agg(first_any_deal_date=('created_time', 'min'))
    .reset_index()
)

# Агрегация по успешным сделкам (Buyer) 
df['is_buyer_deal'] = (df['initial_amount_paid'] > 0) & (df['months_of_study'] > 0)

# Ищем момент, когда лид СТАЛ клиентом (дата первой сделки, где Оплата > 0 и Сервис > 0)
buyers_only = (
    df[df['is_buyer_deal']]
    .groupby('contact_id')
    .agg(first_payment_date=('created_time', 'min'))
    .reset_index()
)
buyers_only['is_buyer'] = True

# Слияние данных 
contacts_update = reg_fixes_all.merge(buyers_only, on='contact_id', how='left')
contacts_update['is_buyer'] = contacts_update['is_buyer'].fillna(False).astype(bool)

# Фикс аномалий регистрации 
if os.path.exists(CONTACTS_CLEAN):
    contacts_reg = pd.read_pickle(CONTACTS_CLEAN)[['id', 'created_time']]
    contacts_update = contacts_update.merge(
        contacts_reg, 
        left_on='contact_id', 
        right_on='id', 
        how='left'
    ).drop(columns='id').rename(columns={'created_time': 'contact_created_time'})
    
    # Считаем аномалией, если любая сделка создана РАНЬШЕ записи о контакте в CRM
    contacts_update['new_registration_date'] = np.where(
        contacts_update['first_any_deal_date'] < contacts_update['contact_created_time'],
        contacts_update['first_any_deal_date'],
        pd.NaT
    )

# Переименовываем contact_id в id для удобного merge
contacts_update = contacts_update.rename(columns={'contact_id': 'id'})

# Сохраняем единый файл
cols_to_export = [
    'id', 'is_buyer', 'first_payment_date', 
    'first_any_deal_date', 'new_registration_date'
]
contacts_update[cols_to_export].to_pickle(BUYERS_INFO_PATH)

n_buyers = contacts_update['is_buyer'].sum()
print(f"Экспортирован файл: {BUYERS_INFO_PATH}")
print(f"Стали покупателями (Оплата + Обучение): {n_buyers}")

Экспортирован файл: ../data/cleaned/buyers_info.pkl
Стали покупателями (Оплата + Обучение): 816

Проверка типов в экспортируемом файле:
id                                Int64
is_buyer                           bool
first_payment_date       datetime64[us]
first_any_deal_date      datetime64[us]
new_registration_date    datetime64[us]
dtype: object


,id,first_any_deal_date,first_payment_date,is_buyer,contact_created_time,new_registration_date
2,5805028000000939010,2024-01-04 08:03:00,2023-07-04 10:11:00,True,2023-07-04 10:11:00,NaT
39,5805028000001350049,2023-07-08 08:56:00,2023-07-08 08:56:00,True,2023-07-08 08:55:00,NaT
67,5805028000001404153,2023-07-10 18:41:00,2024-01-17 19:27:00,True,2023-07-10 18:41:00,NaT
163,5805028000001880249,2023-07-15 13:27:00,2023-07-15 13:27:00,True,2023-07-15 13:27:00,NaT
167,5805028000001882098,2023-07-15 02:16:00,2024-01-28 13:22:00,True,2023-07-15 02:15:00,NaT


## Описание датасета

**Источник:** `Deals (Done).xlsx` — выгрузка потенциальных сделок из CRM  
**Назначение:** основной датасет для анализа выручки, стадий продаж, воронки и LTV

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID сделки в CRM (19-значный) |
| `contact_id` | `int64` | ID связанного контакта (связь с `contacts.id`). Для анонимных сделок = `-1` |
| `deal_owner_name` | `category` | Менеджер, ответственный за ведение сделки |
| `stage` | `category` | Текущая стадия сделки. Статус payment done говорит о том что сделка оплачена. |
| `stage_group` | `category` | **Группировка стадий:** Won/Paid, Lost, Active Sales, Marketing/Lead |
| `initial_amount_paid` | `float64` | **Сумма, которую клиент заплатил первым платежом**  |
| `offer_total_amount` | `float64` | Полная контрактная стоимость продукта |
| `created_time` | `datetime` | Дата и время создания сделки |
| `closing_date` | `datetime` | Дата завершения сделки (факт профита или отказа) |
| `sla` | `Int32` | **Время ответа (мин):** время ответа продажника на заявку контакта |
| `sla_filled` | `Int32` | **Время ответа (мин):** Дозаполненное время ответа продажника на заявку контакта |
| `is_buyer` | `bool` | **Подтверждённый покупатель:** `initial_amount_paid > 0` AND `months_of_study > 0` — оплатил первый взнос И начал обучение. Используется как фильтр для расчёта **Rev** (выручка) и **T** (число транзакций покупателей). |

### Обработка пропусков и специфика колонок
В процессе очистки была проведена работа по восполнению данных (**Backfill**) и нормализации.

- **Нормализация уровня языка (`level_of_deutsch`):**
    - Исходные данные содержали 215 вариантов написания (кириллица, латиница, текст).
    - Применены регулярные выражения и маппинг кириллицы (`а2` -> `A2`, `б1` -> `B1`).
    - Значения приведены к международному стандарту (A0–C2). Неразборчивые записи помечены как `Unclear`.

- **Восполнение (Backfill):** 
    - `source`, `campaign`, `city`, `level_of_deutsch`, `deal_owner_name`: данные "протянуты" между сделками одного и того же клиента (по `contact_id`). Если у клиента в одной сделке был указан уровень языка или город, а в другой — нет, мы восстановили эти данные.
    - `course_duration`: частично восполнено на основе выбранного продукта (`product`).

- **Оставлено "как есть" (NaN):**
    - **Причина:** Данные поля заполняются в CRM преимущественно для успешных сделок (`Won/Paid`). Попытка заполнить их для лидов (через среднее или моду) приведет к серьезному искажению аналитики по продуктовой линейке и LTV. 
    - **closing_date** (~32% пропусков): Отсутствие даты означает, что сделка всё еще находится в работе (не закрыта ни в плюс, ни в минус).

- **Обнаруженные аномалии:**
    - **Успешные сделки с 0 оплатой (28 шт.):** Выявлены сделки в группе `Won/Paid`, где сумма первого платежа равна нулю. Это может указывать на ошибки ввода данных менеджерами (подавляющее число у Kevin Parker - 14). Данные сделки требуют проверки по конкретным менеджерам (см. блок статистики в конце).
    - **~2 399 транзакций с оплатой, но без начала обучения:** `initial_amount_paid > 0` при `months_of_study = 0` → `is_buyer = False`. Эти строки **исключаются из Rev и T** по бизнес-модели. Детали — в блоке диагностики в `07_product_analytics.ipynb`.

- **Заполнение расчетными значениями:**
    - **months_of_study**: для всех сделок в статусе `Lost` проставлено `0`, так как обучения не было.
    - **closing_date**: для сделок в статусе `Lost` с отсутствующей датой закрытия проставлена дата создания (`created_time`), чтобы корректно учитывать их как завершенные в воронке.

- **Заполнено "Unknown":**
    - Категориальные поля (`quality`, `product`, `payment_type` и др.), где отсутствие информации является результатом отсутствия ввода данных в CRM.

**Ключевые связи:**
- `contact_id` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `stage_group` → фильтр для конверсии и ROMI

## Выводы

Датасет сделок содержал несколько слоёв проблем, каждая из которых влияла на достоверность аналитики. Поле `level_of_deutsch` имело **215 вариантов написания** — кириллица, смешанный регистр, свободный текст. CRM не ограничивает ввод, и каждый менеджер записывал уровень языка по-своему. Без нормализации сегментация клиентов по уровню немецкого и продуктовая аналитика по этому срезу были бы просто невозможны.

Выявлено **28 сделок в статусе Won/Paid с нулевой оплатой**, преимущественно у одного менеджера (Kevin Parker — 14 из 28). Это либо ошибки ввода стадии без фиксации платежа, либо намеренная манипуляция показателями конверсии. Без аудита эти сделки искажают и выручку, и конверсию одновременно.

1771 + 7 записей помечено менеджером как "Дубликат". Среди них було обнаружено 2 записи клиентов на 1500. Дубликаты удалены. По 2 записям уточнить.

**33,7% сделок не имеют `closing_date`** — активные или брошенные сделки, которые никто не закрыл в CRM. Принудительное заполнение нулями или медианой исказило бы расчёт длительности сделки и SLA, поэтому для таких записей значение оставлено `NaN`. Отрицательные длительности (`duration_raw < 0`) также сохранены как есть — они являются сигналом ошибки ввода данных конкретным менеджером и используются в аудит-отчёте.

**Что сделано:** `level_of_deutsch` нормализован через regex и маппинг кириллицы до стандарта A0–C2 (Из 215 уникальных значений в сырых данных после приведения осталось нераспознанными только 15 строк); 
пропуски в `source`, `campaign`, `city`, `deal_owner_name` восстановлены по истории контакта (backfill по `contact_id`);
рассчитаны `duration_raw` (с сохранением аномалий) и очищенный `deal_duration_days`- длительность сделки до закрытия (важно для анализа конверсии и деятельности отдела продаж); 
категориальные поля без информации заполнены `'Unknown'`.

---

### Метрики Rev и T: структурная особенность данных

В ходе очистки был установлен критерий `is_buyer = (initial_amount_paid > 0) AND (months_of_study > 0)` — покупатель это тот, кто **оплатил первый взнос И получил сервис** (начал обучение). Это принципиальный момент для двух ключевых финансовых метрик:

- **T (транзакции)** — число сделок с `is_buyer = True`, т.е. подтверждённых платежей c фактом предоставления услуги.  
- **Rev (выручка)** — сумма `initial_amount_paid` только по сделкам с `is_buyer = True`.

Из **3 225 строк с `initial_amount_paid > 0`** только **~826** имеют `is_buyer = True`. Оставшиеся **~2 399 транзакций** (≈74%) — это строки, где оплата есть, но `months_of_study = 0`:  
- клиент заплатил, но по каким-то причинам не приступил к обучению;  
- либо дата старта обучения не была внесена в CRM;  
- либо это промежуточные / отменённые платежи, которые CRM регистрирует как положительные.

Итоговый разрыв: **Rev ≈ 800 K €** (покупатели) vs. **~3,6 M €** (все положительные `initial_amount_paid`). Второй показатель завышен, поскольку включает платежи без подтверждения факта услуги.

По законам евросоюза полученные деньги за непредоставленный сервис возвращаются в полном объеме, поэтому считать их в доход некоректно.

> **Системная рекомендация:** Ввести в CRM обязательное поле **дата начала обучения** при переводе сделки в Won/Paid: сумма платежа > 0 и дата старта обучения обязательны. Дополнительно — еженедельный отчёт по сделкам с `initial_amount_paid > 0` и `months_of_study = 0` для оперативного разбора и доначисления data в CRM. Это позволит сократить разрыв между зарегистрированными платежами и фактической выручкой от подтверждённых покупателей, и сделать метрики Rev и T достоверными, уровень языка клиента из dropdown-списка (не свободный ввод).